<a href="https://colab.research.google.com/github/XGizellHA/Generador-de-Texto-con-PLN/blob/main/PLN_Proyecto_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1.- Instalacion y carga de libreria

In [ ]:
!pip install torch tokenizers pandas matplotlib unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 15.8 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from tokenizers import ByteLevelBPETokenizer
import re
import random

## 2.- Creacion del corpus

In [ ]:
# ============================================================
# 2. Carga de datasets
# ============================================================
# Carga de datasets
songs_df = pd.read_csv("/content/spotify_millsongdata.csv",
                       dtype=str, on_bad_lines="skip", engine="python")
stories_df = pd.read_csv("/content/stories.csv",
                         dtype=str, on_bad_lines="skip", engine="python")

# --- Cuentos ---
stories_texts = stories_df["story"].dropna().tolist()
print("✔ Historias cargadas:", len(stories_texts))

# --- Canciones (solo 1000 aleatorias para acelerar entrenamiento) ---
songs_texts_full = songs_df["text"].dropna().tolist()
random.seed(42)  # reproducibilidad
songs_texts = random.sample(songs_texts_full, 1000)
print("✔ Canciones seleccionadas:", len(songs_texts))


✔ Historias cargadas: 1000
✔ Canciones seleccionadas: 1000


In [ ]:
# ============================================================
# 3. Preprocesamiento: dividir en bloques
# ============================================================
def split_into_blocks(text_list, block_size=50):
    blocks = []
    for t in text_list:
        t = str(t).lower()
        words = re.findall(r"\b\w+\b|[\.,;!?()\"']", t)
        for i in range(0, len(words), block_size):
            blocks.append(words[i:i + block_size])
    return blocks

song_blocks = split_into_blocks(songs_texts, 50)
story_blocks = split_into_blocks(stories_texts, 50)

all_blocks = song_blocks + story_blocks


## 3.- Tokenizacion

In [ ]:
# ============================================================
# 4. Entrenar ByteLevel BPE Tokenizer
# ============================================================
tokenizer = ByteLevelBPETokenizer()
tokenizer.train_from_iterator(
    [" ".join(b) for b in all_blocks],
    vocab_size=40000,
    min_frequency=3,
    special_tokens=["[PAD]", "[UNK]", "[PAR]", "[VERSE]", "[CHORUS]"]
)

vocab_size = tokenizer.get_vocab_size()
PAD_IDX = tokenizer.token_to_id("[PAD]")

# Diccionarios
stoi = tokenizer.get_vocab()          # palabra -> id
itos = {v: k for k, v in stoi.items()} # id -> palabra

print("✔ Vocabulario:", vocab_size)


✔ Vocabulario: 20793


## 4.- Dataset y DataLoader

In [ ]:
# ============================================================
# 5. Dataset y DataLoader
# ============================================================
class TextDataset(Dataset):
    def __init__(self, blocks, tokenizer, max_len):
        self.texts = [" ".join(tok) for tok in blocks]
        self.tokenizer = tokenizer
        self.max_len = max_len + 1

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer.encode(self.texts[idx])
        ids = encoding.ids[:self.max_len]
        # Asegurar que todos los ids < vocab_size
        ids = [min(i, vocab_size-1) for i in ids]
        pad_len = self.max_len - len(ids)
        if pad_len > 0:
            ids += [PAD_IDX]*pad_len
        x = torch.tensor(ids[:-1], dtype=torch.long)
        y = torch.tensor(ids[1:], dtype=torch.long)
        return x, y

def load_mode_dataset(mode="story", batch=32, max_len=64):
    data = story_blocks if mode == "story" else song_blocks
    dataset = TextDataset(data, tokenizer, max_len)
    return DataLoader(dataset, batch_size=batch, shuffle=True)


## 5.- Modelo

In [ ]:
# ============================================================
# 6. Modelo GPT Mini
# ============================================================
class Attention(nn.Module):
    def __init__(self, dim, maxlen, h):
        super().__init__()
        self.h = h
        self.wq = nn.Linear(dim, dim)
        self.wk = nn.Linear(dim, dim)
        self.wv = nn.Linear(dim, dim)
        self.wo = nn.Linear(dim, dim)
        self.scale = (dim // h) ** -0.5
        self.register_buffer("mask", torch.tril(torch.ones(1, 1, maxlen, maxlen)))

    def forward(self, x):
        B, L, D = x.size()
        q = self.wq(x).reshape(B, L, self.h, -1).permute(0, 2, 1, 3)
        k = self.wk(x).reshape(B, L, self.h, -1).permute(0, 2, 3, 1)
        v = self.wv(x).reshape(B, L, self.h, -1).permute(0, 2, 1, 3)
        att = (q @ k) * self.scale
        att = att.masked_fill(self.mask[:, :, :L, :L] == 0, float("-inf"))
        att = att.softmax(dim=-1)
        out = att @ v
        out = out.permute(0, 2, 1, 3).reshape(B, L, D)
        return self.wo(out)

class Block(nn.Module):
    def __init__(self, dim, maxlen, h):
        super().__init__()
        self.att = Attention(dim, maxlen, h)
        self.ln1 = nn.LayerNorm(dim)
        self.ln2 = nn.LayerNorm(dim)
        self.ff = nn.Sequential(
            nn.Linear(dim, 4*dim),
            nn.ReLU(),
            nn.Linear(4*dim, dim)
        )
    def forward(self, x):
        x = x + self.att(x)
        x = self.ln1(x)
        out = x + self.ff(x)
        return self.ln2(out)

class GPT(nn.Module):
    def __init__(self, vocab, maxlen, dim=128, depth=4, heads=4):
        super().__init__()
        self.emb = nn.Embedding(vocab, dim)
        self.pos = nn.Parameter(torch.randn(1, maxlen, dim))
        self.blocks = nn.ModuleList([Block(dim, maxlen, heads) for _ in range(depth)])
        self.fc = nn.Linear(dim, vocab)
    def forward(self, x):
        B, L = x.shape
        x = self.emb(x) + self.pos[:, :L]
        for blk in self.blocks:
            x = blk(x)
        return self.fc(x)


## 6.- Entrenamiento

In [ ]:
# ============================================================
# 7. Entrenamiento
# ============================================================
MAXLEN = 200
BATCH = 64
EPOCHS = 50
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = GPT(vocab_size, MAXLEN).to(device)
opt = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

def train_epoch(model, device, loader, opt):
    model.train()
    total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device).view(-1)
        opt.zero_grad()
        logits = model(x).view(-1, vocab_size)
        loss = loss_fn(logits, y)
        loss.backward()
        opt.step()
        total += loss.item()
    return total / len(loader)


In [ ]:
# --- Entrenamiento cuentos ---
loader_story = load_mode_dataset("story", BATCH, MAXLEN)
for ep in range(EPOCHS):
    print(f"Epoch {ep+1}/{EPOCHS} cuentos ...", end="")
    loss = train_epoch(model, device, loader_story, opt)
    print(f" loss={loss:.4f}")

Epoch 1/50 cuentos ... loss=5.6630
Epoch 2/50 cuentos ... loss=4.4389
Epoch 3/50 cuentos ... loss=3.9738
Epoch 4/50 cuentos ... loss=3.6662
Epoch 5/50 cuentos ... loss=3.4315
Epoch 6/50 cuentos ... loss=3.2421
Epoch 7/50 cuentos ... loss=3.0840
Epoch 8/50 cuentos ... loss=2.9521
Epoch 9/50 cuentos ... loss=2.8388
Epoch 10/50 cuentos ... loss=2.7377
Epoch 11/50 cuentos ... loss=2.6492
Epoch 12/50 cuentos ... loss=2.5701
Epoch 13/50 cuentos ... loss=2.4975
Epoch 14/50 cuentos ... loss=2.4354
Epoch 15/50 cuentos ... loss=2.3749
Epoch 16/50 cuentos ... loss=2.3204
Epoch 17/50 cuentos ... loss=2.2696
Epoch 18/50 cuentos ... loss=2.2225
Epoch 19/50 cuentos ... loss=2.1787
Epoch 20/50 cuentos ... loss=2.1375
Epoch 21/50 cuentos ... loss=2.0990
Epoch 22/50 cuentos ... loss=2.0626
Epoch 23/50 cuentos ... loss=2.0278
Epoch 24/50 cuentos ... loss=1.9973
Epoch 25/50 cuentos ... loss=1.9650
Epoch 26/50 cuentos ... loss=1.9365
Epoch 27/50 cuentos ... loss=1.9088
Epoch 28/50 cuentos ... loss=1.8831
E

In [ ]:
# --- Entrenamiento canciones ---
loader_song = load_mode_dataset("song", BATCH, MAXLEN)
for ep in range(EPOCHS):
    print(f"Epoch {ep+1}/{EPOCHS} canciones ...", end="")
    loss = train_epoch(model, device, loader_song, opt)
    print(f" loss={loss:.4f}")

Epoch 1/50 canciones ... loss=6.2414
Epoch 2/50 canciones ... loss=5.2202
Epoch 3/50 canciones ... loss=4.9459
Epoch 4/50 canciones ... loss=4.7164
Epoch 5/50 canciones ... loss=4.4949
Epoch 6/50 canciones ... loss=4.2734
Epoch 7/50 canciones ... loss=4.0515
Epoch 8/50 canciones ... loss=3.8375
Epoch 9/50 canciones ... loss=3.6323
Epoch 10/50 canciones ... loss=3.4343
Epoch 11/50 canciones ... loss=3.2468
Epoch 12/50 canciones ... loss=3.0757
Epoch 13/50 canciones ... loss=2.9185
Epoch 14/50 canciones ... loss=2.7714
Epoch 15/50 canciones ... loss=2.6332
Epoch 16/50 canciones ... loss=2.5026
Epoch 17/50 canciones ... loss=2.3841
Epoch 18/50 canciones ... loss=2.2712
Epoch 19/50 canciones ... loss=2.1630
Epoch 20/50 canciones ... loss=2.0697
Epoch 21/50 canciones ... loss=1.9701
Epoch 22/50 canciones ... loss=1.8870
Epoch 23/50 canciones ... loss=1.7959
Epoch 24/50 canciones ... loss=1.7214
Epoch 25/50 canciones ... loss=1.6519
Epoch 26/50 canciones ... loss=1.5773
Epoch 27/50 canciones

## 7.- Generacion de texto

In [ ]:
# ============================================================
# 8. Generación de texto optimizada
# ============================================================

import re
import torch
import torch.nn.functional as F

MAXLEN = 200          # longitud máxima de entrada para el modelo
MAX_WORDS = 150       # máximo de palabras en el texto final

# Función de limpieza y post-procesamiento
def clean_generated_text(text):
    # Reemplaza tokens desconocidos y subpalabras
    text = text.replace("[UNK]", "something")
    text = text.replace("Ġ", " ")

    # Elimina espacios antes de signos de puntuación
    text = re.sub(r'\s+([.,!?;:])', r'\1', text)

    # Normaliza espacios
    text = re.sub(r'\s+', ' ', text).strip()

    # Elimina repeticiones consecutivas de palabras
    words = text.split()
    clean_words = []
    prev_word = None
    for w in words:
        if w != prev_word:
            clean_words.append(w)
        prev_word = w
    text = " ".join(clean_words)

    # Truncar al último punto para cerrar la historia
    sentences = re.findall(r'.+?[.!?]', text)
    if sentences:
        text = ' '.join(sentences)

    # Limitar a MAX_WORDS palabras
    words = text.split()
    if len(words) > MAX_WORDS:
        text = " ".join(words[:MAX_WORDS])

    return text

# Función de generación de texto
def generate_text(model, prompt, max_new_tokens, device, tokenizer, temperature=0.6, top_p=0.9):
    model.eval()
    encoding = tokenizer.encode(prompt.lower())
    ids = encoding.ids
    x = torch.tensor([ids], dtype=torch.long).to(device)

    for _ in range(max_new_tokens):
        with torch.no_grad():
            input_to_model = x[:, -MAXLEN:] if x.shape[1] > MAXLEN else x
            logits = model(input_to_model)[:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)

            # Filtrado top-p
            sorted_probs, sorted_indices = torch.sort(probs, descending=True)
            cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
            sorted_indices_to_remove = cumulative_probs > top_p
            sorted_indices_to_remove[..., 0] = 0
            sorted_probs[sorted_indices_to_remove] = 0
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)

            # Elegir token siguiente
            next_id = torch.multinomial(sorted_probs, 1)
            next_id = sorted_indices.gather(-1, next_id)
            x = torch.cat([x, next_id], dim=1)

            # NOTA: Se elimina la condición del token [END] para evitar errores

    text = tokenizer.decode(x[0].tolist())
    text = clean_generated_text(text)
    return text


## 8.- Ejemplos

In [ ]:
# --- Generación de cuento ---
prompt_story = "In a distant galaxy, a young wizard discovers ancient magic in a hidden forest"
story = generate_text(model, prompt_story, max_new_tokens=200, device=device, tokenizer=tokenizer)
print("\n=== Generación de cuento ===\n")
print(story)


=== Generación de cuento ===

in a distant galaxy, a young wizard discovers ancient magic in a hidden forest savior, the last man in black stain, a fiddle out the early passing by the royal scam of blue angels look at the hunter i ' ll catch you help me stop now call a chance to ask the deal is won the in calling if them horn man try to my page ' s embrace i breakree upstairs rings on alright mage use a nerve for arrange cloudedot you gave the door by ' them in the real in call slowed brother the real guy ' rice the shoesding nuclear guy and a work the moment when you ' to my home would be apart and so check ' everything endless my brain ' t see but the teacher "erm from every time just 'r harris cause dem play just at ' hug ' cause


In [ ]:
# --- Generación de canción ---
prompt_song = "Verse 1: In the night I see the stars shining bright"
song = generate_text(model, prompt_song, max_new_tokens=200, device=device, tokenizer=tokenizer)
print("\n=== Generación de canción ===\n")
print(song)



=== Generación de canción ===

verse 1: in the night i see the stars shining bright in their eyes have dust at me ' cause every day i die nine when my life i have no weary, you see no one to entertain days me, no one said no one can and time but like i ' ll leave we can.  and i ' s nowhere to fear and bye.  hell ground?
